# E30 公开调查数据筛选与缺失值有限核对

## tl;dr
按更正附件代码的标签筛选得到1534人；五档频数61/182/762/408/121，与原表相符。完整模型字段的完整行1531，三条伴侣教育“不知道”转为未映射缺失。这是汇总与缺失核对，不是收入效应模型复现，也不推论因果、个人体验或现行诊疗。

## Context & Methods
数据是CRAN TH.data公开源仓库的`inst/rda/CHFLS.rda`，固定提交3bf0832f84f436fe4c57f6cc0de08e298c8470b8；处理依据[更正附件R代码](https://cran.r-project.org/web/packages/multcomp/vignettes/chfls1.R)。原调查1999—2000年，下载2026-11-04；不把软件包当前版本当调查时间。

### Key Assumptions
只复现筛选、频数与模型字段缺失。代码`ZW04 != yes`允许七条legit skip与一条dont know通过，不能将其改称所有人均明确报告异性恋。原R代码对三种无关字段的重复标签警告忽略；本记录在转换因子时保留值标签、去重类别表，所用字段标签无此重复问题。数值除以标准差不改变缺失，故本次只审计缩放前字段，未执行缩放、拟合、AIC选择或同时推断。

## Data
运行需要Python、numpy、pandas、rdata、xarray；`pip install numpy pandas rdata xarray`。默认将公开数据下载到inputs/CHFLS.rda并验证SHA256；可用CHFLS_RDA指定已有副本、CHFLS_PYTHON_DEPS指定依赖目录。数据文件不随本记录发布，代码只打印汇总，不显示个体行。


### 1. 设置依赖


In [1]:
import sys, json
from pathlib import Path
import os, hashlib, urllib.request
if os.environ.get('CHFLS_PYTHON_DEPS'):
    sys.path.insert(0, os.environ['CHFLS_PYTHON_DEPS'])
import numpy as np
import pandas as pd
import rdata



### 2. 校验文件并转换标签


In [2]:
constructors = dict(rdata.conversion.DEFAULT_CLASS_MAP)
def factor_values(obj, attrs, ordered=False):
    levels = list(attrs['levels'])
    values = [None if np.ma.is_masked(i) or i < 1 else levels[int(i)-1] for i in obj]
    return pd.Categorical(values, categories=list(dict.fromkeys(levels)), ordered=ordered)
constructors['factor'] = factor_values
constructors['ordered'] = lambda obj, attrs: factor_values(obj, attrs, True)
data_path = Path(os.environ.get('CHFLS_RDA', 'inputs/CHFLS.rda'))
source_url = 'https://raw.githubusercontent.com/cran/TH.data/3bf0832f84f436fe4c57f6cc0de08e298c8470b8/inst/rda/CHFLS.rda'
if not data_path.exists():
    data_path.parent.mkdir(parents=True, exist_ok=True)
    data_path.write_bytes(urllib.request.urlopen(source_url, timeout=30).read())
assert hashlib.sha256(data_path.read_bytes()).hexdigest() == '38e0b90261ba214abd4edb7654ae7224c51842f0b2ce85150b4e66b68bda8dcc'
raw = rdata.read_rda(data_path, constructor_dict=constructors)['chfls1']


### 3. 按附件建立筛选与字段


In [3]:
orgasm_levels = ['never','rarely','sometimes','often','always']
keep = (raw.ZJ05 == 'female') & (raw.ZW04 != 'yes') & raw.AX16.isin(orgasm_levels)
cohort = raw.loc[keep].copy()
education_levels = ['univ/grad','j col','up mid','low mid','primary','no school']
def rank(series, levels):
    return series.astype(object).map({v: i+1 for i,v in enumerate(levels)}).astype(float)
frame = pd.DataFrame(index=cohort.index)
frame['orgasm'] = rank(cohort.AX16, orgasm_levels)
frame['AincomeSD'] = pd.to_numeric(cohort.INCAM, errors='coerce')
frame['AheightSD'] = pd.to_numeric(cohort.A14, errors='coerce').where(cohort.A14 >= 0)
frame['RAdurationSD'] = pd.to_numeric(cohort.ZJ06, errors='coerce') - pd.to_numeric(cohort.A35, errors='coerce')
frame['RageSD'] = pd.to_numeric(cohort.ZJ06, errors='coerce')
frame['Redu'] = rank(cohort.ZJ07, education_levels)
frame['edudiffSD'] = rank(cohort.A21, education_levels) - frame.Redu
frame['wealthdiffSD'] = pd.to_numeric(cohort.INCRM, errors='coerce') - pd.to_numeric(cohort.INCAM, errors='coerce')
frame['Rhealth'] = rank(cohort.JK01, ['poor','not good','fair','good','excellent'])
frame['Rhappy'] = rank(cohort.JK20, ['v unhappy','not too','relatively','very'])
frame['Region'] = rank(cohort.REGION6, ['CentralW','Northeast','North','InlandS','CoastalE','CoastalS'])


## Results
### 4. 频数及缺失汇总


In [4]:
counts = {v: int((cohort.AX16 == v).sum()) for v in orgasm_levels}
out = {
 'raw_shape': list(raw.shape), 'filtered_n': len(cohort), 'frequencies': counts,
 'high_two_n': counts['often']+counts['always'],
 'high_two_percent': 100*(counts['often']+counts['always'])/len(cohort),
 'missing_by_full_model_field': {c:int(n) for c,n in frame.isna().sum().items()},
 'full_model_complete_n': int(frame.notna().all(axis=1).sum()),
 'partner_education_unmapped': {str(k):int(v) for k,v in cohort.loc[frame.edudiffSD.isna(),'A21'].value_counts().items() if v},
 'initial_model_complete_n': int(frame[['orgasm','AincomeSD','AheightSD']].notna().all(axis=1).sum()),
 'independent_frequency_sum': sum(counts.values()),
 'filter_label_counts': {str(k):int(v) for k,v in cohort.ZW04.value_counts().items() if v},
}
assert sum(counts.values()) == len(cohort)
print(json.dumps(out,indent=2))


{
  "raw_shape": [
    3821,
    423
  ],
  "filtered_n": 1534,
  "frequencies": {
    "never": 61,
    "rarely": 182,
    "sometimes": 762,
    "often": 408,
    "always": 121
  },
  "high_two_n": 529,
  "high_two_percent": 34.485006518904825,
  "missing_by_full_model_field": {
    "orgasm": 0,
    "AincomeSD": 0,
    "AheightSD": 0,
    "RAdurationSD": 0,
    "RageSD": 0,
    "Redu": 0,
    "edudiffSD": 3,
    "wealthdiffSD": 0,
    "Rhealth": 0,
    "Rhappy": 0,
    "Region": 0
  },
  "full_model_complete_n": 1531,
  "partner_education_unmapped": {
    "dont know": 3
  },
  "initial_model_complete_n": 1534,
  "independent_frequency_sum": 1534,
  "filter_label_counts": {
    "no": 1526,
    "legit skip": 7,
    "dont know": 1
  }
}


## Takeaways
原表频数、1534样本及三条教育差缺失在此公开副本上得到对应。1534与1531属于不同字段要求下的分母，不能继续笼统称所有模型无缺失；这也不说明所有历史分析使用完全同一版本。

执行验证：四个代码单元已在同一全新Python命名空间按顺序实际执行，保存真实标准输出，另与独立汇总脚本一致。JSON单元/输出结构已检查。当前环境未装Jupyter内核、nbformat/nbclient/nbconvert，未做Jupyter执行或笔记本界面视觉检查。复核环境可安装这些包后运行`jupyter nbconvert --execute --to notebook --inplace E30_CHFLS筛选与缺失有限核对.ipynb`。不称完整R模型重跑通过。
